In [3]:
# ============ Cell 1: imports & config ============
import os
import sys
from typing import Optional
import shutil

from win32com.client import Dispatch

print("Python:", sys.version.split()[0])

INPUT_FOLDER  = "D:\RClimat\новые"
OUTPUT_FOLDER = "D:\RClimat\готовые"
ROW_NUMBER    = 10
LANDSCAPE     = True
PRINT_AREA    = None
SHEET_NAME    = None   # или "TDSheet"

EXCEL_EXTS = {".xls", ".xlsx", ".xlsm", ".xlsb", ".xltx", ".xltm", ".xlt"}

os.makedirs(INPUT_FOLDER,  exist_ok=True)
os.makedirs(OUTPUT_FOLDER, exist_ok=True)
print("OK: настройки загружены")

Python: 3.9.17
OK: настройки загружены


In [4]:
# ============ Ячейка 2: все функции ============
import os
import re
import shutil
from datetime import datetime, date
from typing import Optional

import pandas as pd
from openpyxl import load_workbook
from win32com.client import Dispatch

# --- страховка: если ячейку 1 забыли выполнить ---
EXCEL_EXTS = globals().get("EXCEL_EXTS",
    {".xls", ".xlsx", ".xlsm", ".xlsb", ".xltx", ".xltm", ".xlt"})


# ================================================================
#  1. Поиск файлов
# ================================================================
def discover_excel_files(folder: str) -> list:
    """Все Excel-файлы в папке (без временных ~$...)."""
    folder = os.path.abspath(folder)
    if not os.path.isdir(folder):
        raise NotADirectoryError(folder)
    result = []
    for name in sorted(os.listdir(folder)):
        if name.startswith("~$"):
            continue
        path = os.path.join(folder, name)
        if os.path.isfile(path) and os.path.splitext(name)[1].lower() in EXCEL_EXTS:
            result.append(path)
    return result


# ================================================================
#  2. Высота строки (точный замер через AutoFit temp-ячейки)
# ================================================================
def measure_text_height(ws, text, width_pt, ref_cell=None):
    """Точная высота текста (pt) при заданной ширине колонки."""
    used = ws.UsedRange
    temp_row = used.Row + used.Rows.Count + 10
    temp_col = used.Column + used.Columns.Count + 10
    if temp_row > 1048576:
        temp_row = 1
    if temp_col > 16384:
        temp_col = 1

    orig_col_width  = ws.Columns(temp_col).ColumnWidth
    orig_row_height = ws.Rows(temp_row).RowHeight

    try:
        cw = max(1.0, (width_pt / 0.75 - 5) / 7)
        ws.Columns(temp_col).ColumnWidth = cw
        cell = ws.Cells(temp_row, temp_col)
        cell.WrapText = True
        if ref_cell is not None:
            try:
                cell.Font.Name   = ref_cell.Font.Name
                cell.Font.Size   = ref_cell.Font.Size
                cell.Font.Bold   = ref_cell.Font.Bold
                cell.Font.Italic = ref_cell.Font.Italic
            except Exception:
                pass
        cell.Value = text
        ws.Rows(temp_row).AutoFit()
        height = ws.Rows(temp_row).RowHeight
    finally:
        try:
            ws.Cells(temp_row, temp_col).ClearContents()
            ws.Columns(temp_col).ColumnWidth = orig_col_width
            ws.Rows(temp_row).RowHeight      = orig_row_height
        except Exception:
            pass
    return height


def fix_row_height(ws, row_number=10, min_h=15.0, max_h=409.0, safety=2.0):
    """Подгоняет высоту строки под её содержимое (merge, многострочный текст)."""
    #print(f"\n=== Fix row {row_number} на листе '{ws.Name}' ===")
    row = ws.Rows(row_number)

    last_col = ws.Cells(row_number, ws.Columns.Count).End(-4159).Column
    if last_col < 1:
        last_col = 1
    #print(f"  last_col = {last_col}")

    max_height = min_h
    processed = set()

    for c in range(1, last_col + 1):
        cell = ws.Cells(row_number, c)

        if cell.MergeCells:
            addr = cell.MergeArea.Address
            if addr in processed:
                continue
            processed.add(addr)
            vc = cell.MergeArea.Cells(1, 1)
            width_pt = cell.MergeArea.Width
            cell.MergeArea.WrapText = True
        else:
            vc = cell
            width_pt = cell.Width
            cell.WrapText = True

        if vc.Value is None:
            continue

        text = str(vc.Value).replace("\r\n", "\n")
        for tag in ("<br>", "<br/>", "<br />", "<BR>", "<BR/>", "<BR />"):
            text = text.replace(tag, "\n")

        h = measure_text_height(ws, text, width_pt, vc)
        #print(f"  {vc.Address}: width={width_pt:.0f}pt "
              #f"len={len(text)} → measured={h:.0f}pt")
        max_height = max(max_height, h)

    target = min(max_h, max(max_height + safety, min_h))
    row.RowHeight = target
    #print(f"  ИТОГО RowHeight = {target:.0f}pt")
    return target


# ================================================================
#  3. Конвертация в .xlsx + форматирование
# ================================================================
def process_excel_file(xlApp, src_path, out_folder, row_number=10,
                       landscape=True, print_area=None, sheet_name=None):
    """Открыть любой Excel → SaveAs .xlsx под именем из C1 → форматирование."""
    src_path   = os.path.abspath(src_path)
    out_folder = os.path.abspath(out_folder)
    os.makedirs(out_folder, exist_ok=True)

    wb = xlApp.Workbooks.Open(src_path)
    try:
        ws = wb.Worksheets(1) if sheet_name is None else wb.Worksheets(sheet_name)

        # --- имя нового файла = значение ячейки C1 (например "21-368557") ---
        c1 = ws.Cells(1, 3).Value
        if c1 is not None and str(c1).strip():
            base = str(c1).strip()
        else:
            base = os.path.splitext(os.path.basename(src_path))[0]
            print(f"  [i] C1 пуст — берём имя исходника: {base}")

        # санитизация под Windows
        base = re.sub(r'[\\/:*?"<>|]', '_', base).strip() or "file"
        # убираем расширение, если вдруг уже есть
        if base.lower().endswith((".xlsx", ".xls")):
            base = os.path.splitext(base)[0]

        out_path = os.path.join(out_folder, base + ".xlsx")

        # защита от перезаписи исходника (если папки совпадают)
        if src_path.lower() == out_path.lower():
            out_path = os.path.join(out_folder, base + "_formatted.xlsx")

        # --- SaveAs ---
        wb.SaveAs(out_path, 51)          # 51 = xlOpenXMLWorkbook

        # --- форматирование ---
        fix_row_height(ws, row_number)

        ps = ws.PageSetup
        ps.Orientation        = 1 if landscape else 1
        ps.PaperSize          = 9
        ps.Zoom               = False
        ps.FitToPagesWide     = 1
        ps.FitToPagesTall     = 1
        ps.CenterHorizontally = True
        ps.CenterVertically   = True
        if print_area:
            ps.PrintArea = print_area

        wb.Save()
    finally:
        wb.Close(False)

    return out_path

def process_folder(input_folder, output_folder, row_number=10,
                   landscape=True, print_area=None, sheet_name=None):
    """Обрабатывает все Excel-файлы папки. Возвращает (ok, failed)."""
    files = discover_excel_files(input_folder)
    print(f"Найдено файлов: {len(files)}")
    if not files:
        return [], []

    ok, failed = [], []
    xlApp = Dispatch('Excel.Application')
    xlApp.Visible       = False
    xlApp.DisplayAlerts = False
    try:
        for i, src in enumerate(files, 1):
            name = os.path.basename(src)
            #print(f"[{i}/{len(files)}] {name}", end=" ... ")
            try:
                out = process_excel_file(
                    xlApp, src, output_folder,
                    row_number=row_number,
                    landscape=landscape,
                    print_area=print_area,
                    sheet_name=sheet_name,
                )
                #print(f"OK → {os.path.basename(out)}")
                ok.append((src, out))
            except Exception as e:
                print(f"ОШИБКА: {e}")
                failed.append((name, str(e)))
    finally:
        xlApp.Quit()
        del xlApp

    print("\n" + "=" * 50)
    print(f"Успешно: {len(ok)} / {len(files)}")
    if failed:
        print("Не удалось обработать:")
        for name, err in failed:
            print(f"  • {name}: {err}")
    return ok, failed


# ================================================================
#  4. Сортировка файлов: ПЭ / Реестры / Неизвестно
# ================================================================
KEY_PE     = "поручение экспедитору"
KEY_REESTR = "реестр документов"


def detect_type_by_filename(name):
    low = name.lower()
    if "реестр" in low:
        return "реестр"
    if "поручение" in low:
        return "ПЭ"
    return None


def detect_type_by_content(xlApp, path):
    try:
        wb = xlApp.Workbooks.Open(Filename=path, ReadOnly=True, UpdateLinks=0)
    except Exception as e:
        print(f"(open fail: {e})", end=" ")
        return None
    try:
        ws = wb.Worksheets(1)
        parts = []
        for r in range(1, 21):
            for c in range(1, 4):
                v = ws.Cells(r, c).Value
                if v is not None:
                    parts.append(str(v))
        text = " ".join(parts).lower()
        if KEY_REESTR in text:
            return "реестр"
        if KEY_PE in text:
            return "ПЭ"
        return None
    finally:
        wb.Close(False)


def sort_files_by_type(input_folder, pe_folder=None,
                       reestr_folder=None, unknown_folder=None):
    """Раскладывает Excel-файлы по папкам: ПЭ / Реестры / Неизвестно."""
    input_folder   = os.path.abspath(input_folder)
    pe_folder      = os.path.abspath(pe_folder     or os.path.join(input_folder, "ПЭ"))
    reestr_folder  = os.path.abspath(reestr_folder or os.path.join(input_folder, "Реестры"))
    unknown_folder = os.path.abspath(unknown_folder or os.path.join(input_folder, "Неизвестно"))

    for p in (pe_folder, reestr_folder, unknown_folder):
        os.makedirs(p, exist_ok=True)

    files = discover_excel_files(input_folder)
    print(f"Найдено файлов: {len(files)}")
    if not files:
        return {"ПЭ": 0, "реестр": 0, None: 0}

    xlApp = Dispatch('Excel.Application')
    xlApp.Visible       = False
    xlApp.DisplayAlerts = False
    counters = {"ПЭ": 0, "реестр": 0, None: 0}

    try:
        for i, path in enumerate(files, 1):
            name = os.path.basename(path)
            print(f"[{i}/{len(files)}] {name}", end=" ... ")

            kind, method = detect_type_by_filename(name), "имя"
            if kind is None:
                kind, method = detect_type_by_content(xlApp, path), "содержимое"

            dst = {"ПЭ": pe_folder, "реестр": reestr_folder}.get(kind, unknown_folder)
            counters[kind] = counters.get(kind, 0) + 1

            target = os.path.join(dst, name)
            if os.path.exists(target):
                base, ext = os.path.splitext(name)
                n = 1
                while os.path.exists(target):
                    target = os.path.join(dst, f"{base}_{n}{ext}")
                    n += 1

            shutil.move(path, target)
            print(f"{kind or 'неизвестно'} (по {method}) → {os.path.basename(dst)}/")
    finally:
        xlApp.Quit()
        del xlApp

    print("\n" + "=" * 50)
    print(f"ПЭ:           {counters['ПЭ']}")
    print(f"Реестры:      {counters['реестр']}")
    print(f"Неизвестно:   {counters[None]}")
    return counters


# ================================================================
#  5. Отчёт ПЭ + Реестр → .txt
# ================================================================
def extract_ttn_list(reestr_path):
    """
    Возвращает список ТТН из реестра. Находит колонку по заголовку '№ ТТН',
    затем фильтрует по универсальному паттерну '<префикс>-<>=5 цифр>'.
    """
    df = pd.read_excel(reestr_path, header=None)

    header_row, ttn_col = None, None
    for r in range(min(30, len(df))):
        for c in range(df.shape[1]):
            v = df.iat[r, c]
            if isinstance(v, str) and 'ТТН' in v and '№' in v:
                header_row, ttn_col = r, c
                break
        if ttn_col is not None:
            break

    if ttn_col is None:
        raise ValueError(f"колонка '№ ТТН' не найдена в {os.path.basename(reestr_path)}")

    pattern = re.compile(r'^[A-Za-zА-Яа-яЁё0-9]{1,5}-\d{5,}$')

    vals = df.iloc[header_row + 1:, ttn_col].dropna().astype(str)
    result = []
    for v in vals:
        v = v.strip().rstrip(';,')
        if pattern.match(v):
            result.append(v)
    return result


def build_reestr_index(reestr_folder):
    """Карта plan_number -> путь к реестру (по '№ <число>' в имени файла)."""
    index = {}
    for name in os.listdir(reestr_folder):
        if name.startswith("~$"):
            continue
        if not name.lower().endswith((".xls", ".xlsx")):
            continue
        m = re.search(r'№\s*(\d{4,})', name)
        if m:
            index[m.group(1)] = os.path.join(reestr_folder, name)
    return index


def fmt_date(v):
    """Универсально форматирует дату из Excel (datetime, строка, None)."""
    if v is None:
        return ""
    if isinstance(v, (datetime, date)):
        return v.strftime('%d.%m.%Y')
    return str(v)


def generate_flight_report(pe_path, reestr_path, output_folder):
    """Извлекает поля из ПЭ и ТТН из реестра, пишет .txt."""
    wb = load_workbook(pe_path, data_only=True)
    try:
        sheet = wb.active

        raw_number = sheet["C1"].value
        if raw_number is None:
            raise ValueError("C1 пуст — нет номера ПЭ")
        number_str = str(raw_number).strip()

        date_1 = 'c ' + fmt_date(sheet["E18"].value)[:5] + ' по'
        date_2 = fmt_date(sheet["E19"].value)

        city_raw = sheet["C5"].value
        city = str(city_raw)[6:] if city_raw else ""

        driver_raw = str(sheet["E25"].value or "")
        driver_short = driver_raw[:18]
        driver = 'водитель ' + driver_short + '.,'

        number = 'ПЭ №' + number_str + ', '

        auto_raw = str(sheet["E24"].value or "")
        auto = '  ' + auto_raw[:6]
    finally:
        wb.close()

    ttn_list = extract_ttn_list(reestr_path)
    if not ttn_list:
        raise ValueError(f"ТТН не найдены в {os.path.basename(reestr_path)}")
    ttn = '№ ТТН, ' + ', '.join(ttn_list)

    flight = ' '.join([city, date_1, date_2, driver, number])
    result = flight + ttn

    title = number_str + auto + ' ' + driver_short + '.txt'
    title = re.sub(r'[\\/:*?"<>|]', '_', title).strip()
    out_path = os.path.join(output_folder, title)

    with open(out_path, 'w', encoding='utf-8') as f:
        f.write(result)

    return out_path, result


def generate_all_reports(pe_folder, reestr_folder, output_folder):
    """Для каждого ПЭ находит реестр по плановому номеру и пишет .txt."""
    os.makedirs(output_folder, exist_ok=True)

    reestr_index = build_reestr_index(reestr_folder)
    print(f"Реестров в индексе: {len(reestr_index)}")

    pe_files = discover_excel_files(pe_folder)
    print(f"ПЭ файлов: {len(pe_files)}")
    if not pe_files:
        return [], []

    ok, failed = [], []
    for i, pe_path in enumerate(pe_files, 1):
        name = os.path.basename(pe_path)
        print(f"[{i}/{len(pe_files)}] {name}")

        try:
            wb = load_workbook(pe_path, data_only=True)
            try:
                raw = wb.active["C1"].value
            finally:
                wb.close()
            if raw is None:
                raise ValueError("C1 пуст")
            number_str = str(raw).strip()

            nums = re.findall(r'\d{4,}', number_str)
            if not nums:
                raise ValueError(f"не извлечь план-номер из {number_str!r}")
            plan_num = nums[-1]
            #print(f"    план №: {plan_num}")

            reestr_path = reestr_index.get(plan_num)
            if not reestr_path:
                raise FileNotFoundError(
                    f"реестр для № {plan_num} не найден в {reestr_folder}")
            print(f"    реестр: {os.path.basename(reestr_path)}")

            out_path, result = generate_flight_report(
                pe_path, reestr_path, output_folder)
            print(f"    ✓ {os.path.basename(out_path)}")
            ok.append((pe_path, out_path))

        except Exception as e:
            print(f"    ! {e}")
            failed.append((name, str(e)))

    print("\n" + "=" * 50)
    print(f"Успешно: {len(ok)} / {len(pe_files)}")
    if failed:
        print("Не удалось обработать:")
        for n, err in failed:
            print(f"  • {n}: {err}")
    return ok, failed


print("OK: ячейка 2 загружена — все функции определены")

OK: ячейка 2 загружена — все функции определены


In [6]:
# ============ Ячейка 3: запуск ============
sort_files_by_type(INPUT_FOLDER)

PE_FOLDER     = os.path.join(INPUT_FOLDER, "ПЭ")
REESTR_FOLDER = os.path.join(INPUT_FOLDER, "Реестры")

ok, failed = process_folder(PE_FOLDER, OUTPUT_FOLDER,
                            row_number=ROW_NUMBER,
                            landscape=LANDSCAPE,
                            print_area=PRINT_AREA,
                            sheet_name=SHEET_NAME)

ok_txt, failed_txt = generate_all_reports(OUTPUT_FOLDER, REESTR_FOLDER, OUTPUT_FOLDER)

Найдено файлов: 14
[1/14] 368551.xls ... ПЭ (по содержимое) → ПЭ/
[2/14] 371229.xls ... ПЭ (по содержимое) → ПЭ/
[3/14] 372464.xls ... ПЭ (по содержимое) → ПЭ/
[4/14] 372475.xls ... ПЭ (по содержимое) → ПЭ/
[5/14] 372482.xls ... ПЭ (по содержимое) → ПЭ/
[6/14] 372489.xls ... ПЭ (по содержимое) → ПЭ/
[7/14] 373308.xls ... ПЭ (по содержимое) → ПЭ/
[8/14] Реестр документов к плану № 368551 от 24.09.2026 9_51_58.xls ... реестр (по имя) → Реестры/
[9/14] Реестр документов к плану № 371229 от 09.10.2026 7_32_24.xls ... реестр (по имя) → Реестры/
[10/14] Реестр документов к плану № 372464 от 16.10.2026 16_17_46.xls ... реестр (по имя) → Реестры/
[11/14] Реестр документов к плану № 372475 от 16.10.2026 16_17_46.xls ... реестр (по имя) → Реестры/
[12/14] Реестр документов к плану № 372482 от 16.10.2026 16_17_46.xls ... реестр (по имя) → Реестры/
[13/14] Реестр документов к плану № 372489 от 16.10.2026 16_17_46.xls ... реестр (по имя) → Реестры/
[14/14] Реестр документов к плану № 373308 от 22.1